In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [ ]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "effiecientnet_v2",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [5]:

efficientnet_v2_s = models.efficientnet_v2_s(pretrained=True)
efficientnet_v2_s = models.efficientnet_v2_s(pretrained=True)

for param in efficientnet_v2_s.parameters():
    param.requires_grad = False

in_features = efficientnet_v2_s.classifier[1].in_features
efficientnet_v2_s.classifier[1] = nn.Linear(in_features, 84)

gpu = torch.device("cuda")
efficientnet_v2_s = efficientnet_v2_s.to(gpu)


/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/iztihad/venvs/ml/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_S_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
for param in efficientnet_v2_s.features[-2:].parameters():
    param.requires_grad = True

In [9]:
# for param in mobilenet_v2.features[-6:].parameters():
#     param.requires_grad = True

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": efficientnet_v2_s.classifier[1].parameters(), "lr": 1e-4, "weight-decay": 1e-4},
    {"params": efficientnet_v2_s.features.parameters(), "lr": 1e-5, "weight-decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [7]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [8]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(efficientnet_v2_s, val_dataloader)

        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [15]:
train_model(efficientnet_v2_s, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.08707549542575292, Validation Accuracy: 0.9392001930152603
Epoch: 2, Training Loss: 0.07594203842674284, Validation Accuracy: 0.9376319440255745
Epoch: 3, Training Loss: 0.06855095448013629, Validation Accuracy: 0.9395620966282646
Epoch: 4, Training Loss: 0.06132477654707001, Validation Accuracy: 0.9384160685204174
Epoch: 5, Training Loss: 0.054960900047581446, Validation Accuracy: 0.9378128958320767
Epoch: 6, Training Loss: 0.05145213734484618, Validation Accuracy: 0.9400446347789372
Epoch: 7, Training Loss: 0.048267107918932885, Validation Accuracy: 0.9393208275529285
Epoch: 8, Training Loss: 0.0461319361004932, Validation Accuracy: 0.9399843175101031
Epoch: 9, Training Loss: 0.04216946799899058, Validation Accuracy: 0.9399240002412691
Epoch: 10, Training Loss: 0.03982373931204499, Validation Accuracy: 0.9396224138970988


In [16]:
accuracy = validate_model(efficientnet_v2_s, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 93.58947400042055
